# CM05 berberine Boltz2 Colab run

Purpose: run Boltz2 on the CM05 berberine priority candidates with a GPU-backed Colab runtime. This notebook first performs a one-candidate smoke test, then runs the six-candidate batch if the smoke test produces structure, confidence and affinity files.

Scientific boundary: outputs are computational prioritization results only. They are not experimental binding proof.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json, time, textwrap

REPO_URL = 'https://github.com/lhs1308781604-droid/20260324.git'
BRANCH = 'codex/boltz2-cm05-berberine-20260620'
PKG_REL = 'cloud_run_packages/boltz2_cm05_berberine_priority6_2026-06-20'
WORK = Path('/content/20260324')
PKG = WORK / PKG_REL
RESULT_ZIP = Path('/content/CM05_berberine_Boltz2_Colab_results.zip')

def run(cmd, cwd=None, env=None, check=False):
    print('\n$ ' + ' '.join(map(str, cmd)))
    p = subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, env=env, text=True)
    print('exit_code =', p.returncode)
    if check and p.returncode != 0:
        raise RuntimeError('Command failed: ' + ' '.join(map(str, cmd)))
    return p.returncode

def show_file(path, n=30):
    path = Path(path)
    print(f'--- {path} ---')
    if not path.exists():
        print('MISSING')
        return
    lines = path.read_text(errors='replace').splitlines()
    for line in lines[:n]:
        print(line)
    if len(lines) > n:
        print(f'... {len(lines)-n} more lines')

In [ ]:
# GPU check. This cell must show an NVIDIA GPU before formal Boltz2 execution.
run(['nvidia-smi'])
try:
    import torch
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    print('cuda device count:', torch.cuda.device_count())
except Exception as e:
    print('torch not loaded before install:', e)

In [ ]:
# Clone the prepared input package from GitHub.
if WORK.exists():
    shutil.rmtree(WORK)
run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(WORK)], check=True)
print('Package exists:', PKG.exists())
show_file(PKG / 'tables/cloud_priority6_candidate_manifest.tsv', n=10)

In [ ]:
# Install Boltz2 and package dependencies in the Colab runtime.
run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
run([sys.executable, '-m', 'pip', 'install', '-r', str(PKG / 'requirements.txt')], check=True)
run(['boltz', '--help'], check=True)

In [ ]:
# Smoke test: one candidate, no-template route, GPU, MSA server enabled, shortened sampling.
env = os.environ.copy()
env.update({
    'BOLTZ_RUN_MODE': 'smoke',
    'BOLTZ_ACCELERATOR': 'gpu',
    'BOLTZ_DEVICES': '1',
    'BOLTZ_FORCE_NO_TEMPLATE': '1',
    'BOLTZ_ALLOW_FALLBACK': '1',
    'BOLTZ_USE_MSA_SERVER': '1',
    'BOLTZ_USE_POTENTIALS': '0',
    'BOLTZ_RECYCLING_STEPS': '1',
    'BOLTZ_SAMPLING_STEPS': '5',
    'BOLTZ_SAMPLING_STEPS_AFFINITY': '5',
    'BOLTZ_DIFFUSION_SAMPLES': '1',
    'BOLTZ_DIFFUSION_SAMPLES_AFFINITY': '1',
    'BOLTZ_SEED': '20260620',
})
run([sys.executable, 'scripts/run_boltz2_cloud.py'], cwd=PKG, env=env, check=False)
show_file(PKG / 'cloud_results/status/boltz2_cm05_status.tsv', n=20)
show_file(PKG / 'cloud_results/logs/boltz2_runner.log', n=40)

In [ ]:
# Smoke validation gate. Stop if the smoke test did not produce all expected outputs.
import csv
status_path = PKG / 'cloud_results/status/boltz2_cm05_status.tsv'
rows = list(csv.DictReader(status_path.open(), delimiter='	')) if status_path.exists() else []
if not rows:
    raise RuntimeError('Smoke test did not produce a status table.')
smoke = rows[0]
print(json.dumps(smoke, indent=2))
required = ['cif_exists', 'confidence_exists', 'affinity_exists']
if not all(smoke.get(k) == 'True' for k in required):
    raise RuntimeError('Smoke test failed: missing CIF, confidence, or affinity output. Inspect logs before formal batch.')
print('Smoke test passed. Proceeding to six-candidate formal batch.')

In [ ]:
# Formal six-candidate run. Uses no-template route because the template route failed parser checks in Cloud.
# Parameters are higher than smoke and are intended for candidate ranking, not experimental proof.
env = os.environ.copy()
env.update({
    'BOLTZ_RUN_MODE': 'formal',
    'BOLTZ_ACCELERATOR': 'gpu',
    'BOLTZ_DEVICES': '1',
    'BOLTZ_FORCE_NO_TEMPLATE': '1',
    'BOLTZ_ALLOW_FALLBACK': '1',
    'BOLTZ_USE_MSA_SERVER': '1',
    'BOLTZ_USE_POTENTIALS': '1',
    'BOLTZ_RECYCLING_STEPS': '3',
    'BOLTZ_SAMPLING_STEPS': '200',
    'BOLTZ_SAMPLING_STEPS_AFFINITY': '200',
    'BOLTZ_DIFFUSION_SAMPLES': '1',
    'BOLTZ_DIFFUSION_SAMPLES_AFFINITY': '5',
    'BOLTZ_SEED': '20260620',
})
run([sys.executable, 'scripts/run_boltz2_cloud.py'], cwd=PKG, env=env, check=False)
show_file(PKG / 'cloud_results/status/boltz2_cm05_status.tsv', n=20)
show_file(PKG / 'cloud_results/tables/boltz2_cm05_summary.tsv', n=20)

In [ ]:
# Package results and trigger browser download.
if RESULT_ZIP.exists():
    RESULT_ZIP.unlink()
shutil.make_archive(str(RESULT_ZIP).replace('.zip',''), 'zip', PKG / 'cloud_results')
print('Result zip:', RESULT_ZIP)
print('Zip size MB:', RESULT_ZIP.stat().st_size / 1024 / 1024)
try:
    from google.colab import files
    files.download(str(RESULT_ZIP))
except Exception as e:
    print('Automatic download did not start:', e)
    print('Open the file browser on the left and download:', RESULT_ZIP)